# Extension of the Llama-2-7B vocabulary for the Bashkir language

Goal:

- Train the BPE tokenizer on the `bashqort-raw` corpus (28,600 new tokens).
- Expand the dictionary of the Llama-2-7B model by adding Bashkir tokens.
- Perform a **short additional training** (50 steps) on new embeddings to show that they are integrated. That gives each new token some approximate value, so that later long training begins not from scratch, but from tokens that already “approximately understandable”.
- Save the extended model and tokenizer for use in Experiments A and B.

## Imports & Configs

In [1]:
!pip install -q wandb sentencepiece tokenizers datasets accelerate peft bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 27.6 MB/s eta 0:00:00


In [2]:
import os
import torch
import wandb
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, Trainer, TrainingArguments
)
from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
import numpy as np
from tqdm import tqdm

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: False


In [3]:
import wandb

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()


wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: e278979 (e278979-metu-middle-east-technical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
from huggingface_hub import login, whoami
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF_TOKEN_METU"))

## Load Dataset

In [5]:
from datasets import load_dataset

dataset = load_dataset("metuKKhud/bashqort-raw")

README.md: 0.00B [00:00, ?B/s]

data/bash_news_articles-00000-of-00001.p(…):   0%|          | 0.00/38.5M [00:00<?, ?B/s]

data/bashgazet_articles-00000-of-00001.p(…):   0%|          | 0.00/1.46M [00:00<?, ?B/s]

data/neftcity_articles-00000-of-00001.pa(…):   0%|          | 0.00/451k [00:00<?, ?B/s]

data/public_domain-00000-of-00001.parque(…):   0%|          | 0.00/66.6k [00:00<?, ?B/s]

...

Generating texts_jeshlek split:   0%|          | 0/6259 [00:00<?, ? examples/s]

Generating texts_kiskeufa split:   0%|          | 0/45 [00:00<?, ? examples/s]

Generating texts_kulturarb split:   0%|          | 0/1557 [00:00<?, ? examples/s]

Generating texts_president_rb split:   0%|          | 0/1879 [00:00<?, ? examples/s]

Generating texts_tabin split:   0%|          | 0/539 [00:00<?, ? examples/s]

In [6]:
clean_texts = []
for split_name, split_dataset in dataset.items():
    # is_shuffled == False
    clean_split = split_dataset.filter(lambda x: x['is_shuffled'] is False)
    if len(clean_split) != 0:
        print("Non-shuffled split:", split_name)
    clean_texts.extend(clean_split['text'])

print(f"Overall not-shuffled: {len(clean_texts)}")

Filter:   0%|          | 0/61932 [00:00<?, ? examples/s]

Non-shuffled split: bash_news_articles


Filter:   0%|          | 0/803 [00:00<?, ? examples/s]

Non-shuffled split: bashgazet_articles


Filter:   0%|          | 0/530 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/45 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1557 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1879 [00:00<?, ? examples/s]

Filter:   0%|          | 0/539 [00:00<?, ? examples/s]

Overall not-shuffled: 63270


## Train Tokenizer

In [7]:
def train_bpe_tokenizer(texts, vocab_size=28600):
    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
    tokenizer.decoder = decoders.ByteLevel()
    
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"],
        min_frequency=2,
        show_progress=True
    )
    tokenizer.train_from_iterator(texts, trainer=trainer)
    return tokenizer

print("Training BPE...")
bpe_tokenizer = train_bpe_tokenizer(clean_texts, vocab_size=28600)
bpe_tokenizer.save("bashkir_bpe_tokenizer.json")

Training BPE...





In [8]:
test_text = "миҙгелдә"
encoded = bpe_tokenizer.encode(test_text)
decoded = bpe_tokenizer.decode(encoded.ids)
print(decoded)

 миҙгелдә


In [9]:
test_paragraph = """
Башҡортостанда йәйге каникулдар башланды. Мәктәп уҡыусылары өсөн төрлө ял лагерҙары эшләй. 
Өфөлә балалар өсөн бушлай мастер-кластар ойошторола. Спорт ярыштары, конкурстар һәм экскурсиялар планлаштырыла.
Ата-әсәләр балаларының хәүефһеҙлеген тәьмин итергә тейеш.
"""

unique_chars = set(test_paragraph)
print("Unique symbols:", unique_chars)

problem_chars = []
for ch in unique_chars:
    enc_ch = bpe_tokenizer.encode(ch)
    if '[UNK]' in enc_ch.tokens:
        problem_chars.append(ch)
        print(f"Problem symb: '{ch}' (code U+{ord(ch):04X})")

if not problem_chars:
    print("Ok")
else:
    print(f"Found {len(problem_chars)} problem symbols: {problem_chars}")

Unique symbols: {'Ө', 'ь', 'ф', 'о', 'у', 'ң', 'С', 'и', 'а', 'А', 'ҡ', 'м', 'ө', 'М', 'я', 'с', 'ә', 'р', 'е', 'б', 'ы', 'ш', 'т', 'э', 'н', 'й', 'г', 'п', 'ү', 'х', '.', 'Б', ',', 'һ', 'ҙ', '-', 'д', 'к', '\n', 'л', ' '}
Problem symb: '
' (code U+000A)
Found 1 problem symbols: ['\n']


In [10]:
test_paragraph_clean = test_paragraph.replace('\n', ' ')
encoded = bpe_tokenizer.encode(test_paragraph_clean)
print(f"[UNK]: {'[UNK]' in encoded.tokens}")
decoded = bpe_tokenizer.decode(encoded.ids)
assert decoded.strip() == test_paragraph_clean.strip()
print("✅")

[UNK]: False
✅
